# HumanEval Multi-Seed Replication

Select **Runtime > Change runtime type > T4 GPU**, choose seed `7` or `123` from the form, then run the single cell. Use a fresh runtime for each seed.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import files as colab_files

REPOSITORY = 'https://github.com/Hamza-Nadif/code-grpo-humaneval.git'
SEED = 7  # @param [7, 123]
WORKDIR = Path('/content/code-sft-grpo-multiseed')
MODEL = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
BASELINE_DIR = Path(f'results/base-heldout-seed-{SEED}')
SFT_DIR = Path(f'outputs/qwen-code-sft-seed-{SEED}')
SFT_RESULTS = Path(f'results/sft-heldout-seed-{SEED}')
GRPO_DIR = Path(f'outputs/qwen-code-sft-grpo-seed-{SEED}')
GRPO_RESULTS = Path(f'results/sft-grpo-heldout-seed-{SEED}')
BUNDLE_DIR = Path(f'artifacts/sft-grpo-seed-{SEED}')

def run(command):
    print('\n$', ' '.join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], check=True)

def evaluate(adapter, output_dir):
    command = [
        sys.executable, 'evaluate_baseline.py',
        '--data', 'data/humaneval_test.jsonl',
        '--backend', 'transformers',
        '--model', MODEL,
        '--quantization', '4bit',
        '--samples-per-task', '1',
        '--max-new-tokens', '256',
        '--temperature', '0',
        '--seed', str(SEED),
        '--executor', 'local',
        '--allow-local-code-execution',
        '--output-dir', output_dir,
    ]
    if adapter is not None:
        command.extend(['--adapter', adapter])
    run(command)

def copy_final_adapter(source, destination):
    destination.mkdir(parents=True, exist_ok=True)
    for path in source.iterdir():
        if path.is_file():
            shutil.copy2(path, destination / path.name)

print('STEP 1/10 - Checking GPU and storage', flush=True)
run(['nvidia-smi'])
free_gb = shutil.disk_usage('/content').free / 2**30
print(f'Free Colab storage: {free_gb:.1f} GB', flush=True)
if free_gb < 15:
    raise RuntimeError('At least 15 GB of free Colab storage is required.')

print('STEP 2/10 - Loading the latest project version', flush=True)
if WORKDIR.exists():
    run(['git', '-C', WORKDIR, 'pull', '--ff-only', 'origin', 'main'])
else:
    run(['git', 'clone', '--branch', 'main', REPOSITORY, WORKDIR])
os.chdir(WORKDIR)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'Using Git commit: {commit}', flush=True)

print('STEP 3/10 - Installing dependencies and preparing disjoint data', flush=True)
run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', 'requirements.txt', '-r', 'requirements-dev.txt',
])
run([sys.executable, 'build_training_data.py', '--output-dir', 'data'])
run([sys.executable, '-m', 'pytest', '-q'])

print('STEP 4/10 - Evaluating the frozen base model', flush=True)
evaluate(None, BASELINE_DIR)

print('STEP 5/10 - Training the completion-only SFT adapter', flush=True)
run([
    sys.executable, 'train_sft.py',
    '--model', MODEL,
    '--train-data', 'data/humaneval_train.jsonl',
    '--eval-data', 'data/humaneval_validation.jsonl',
    '--quantization', '4bit',
    '--precision', 'fp16',
    '--max-steps', '30',
    '--gradient-accumulation-steps', '4',
    '--max-length', '768',
    '--seed', str(SEED),
    '--output-dir', SFT_DIR,
])

print('STEP 6/10 - Evaluating SFT on the held-out tasks', flush=True)
evaluate(SFT_DIR, SFT_RESULTS)

print('STEP 7/10 - Continuing the SFT adapter with GRPO', flush=True)
run([
    sys.executable, 'train_grpo.py',
    '--model', MODEL,
    '--adapter', SFT_DIR,
    '--train-data', 'data/humaneval_train.jsonl',
    '--eval-data', 'data/humaneval_validation.jsonl',
    '--quantization', '4bit',
    '--precision', 'fp16',
    '--num-generations', '4',
    '--gradient-accumulation-steps', '4',
    '--max-completion-length', '256',
    '--max-steps', '50',
    '--seed', str(SEED),
    '--executor', 'local',
    '--allow-local-code-execution',
    '--output-dir', GRPO_DIR,
])

print('STEP 8/10 - Evaluating SFT+GRPO on the held-out tasks', flush=True)
evaluate(GRPO_DIR, GRPO_RESULTS)

print('STEP 9/10 - Building the three-stage comparison', flush=True)
base = json.loads((BASELINE_DIR / 'summary.json').read_text())
sft = json.loads((SFT_RESULTS / 'summary.json').read_text())
grpo = json.loads((GRPO_RESULTS / 'summary.json').read_text())
base_score = base['metrics']['pass@1']
sft_score = sft['metrics']['pass@1']
grpo_score = grpo['metrics']['pass@1']
sft_config = json.loads((SFT_DIR / 'experiment_config.json').read_text())
grpo_state_path = GRPO_DIR / 'checkpoint-50' / 'trainer_state.json'
grpo_state = json.loads(grpo_state_path.read_text()) if grpo_state_path.exists() else {}
grpo_logs = [
    row for row in grpo_state.get('log_history', [])
    if 'reward' in row and 'eval_reward' not in row
]
zero_gradient_steps = sum(float(row.get('grad_norm', 0.0)) == 0.0 for row in grpo_logs)
mean_clipped_ratio = (
    sum(float(row.get('completions/clipped_ratio', 0.0)) for row in grpo_logs) / len(grpo_logs)
    if grpo_logs else None
)
comparison = {
    'git_commit': commit,
    'seed': SEED,
    'model': MODEL,
    'held_out_tasks': base['tasks'],
    'base_pass_at_1': base_score,
    'sft_pass_at_1': sft_score,
    'sft_grpo_pass_at_1': grpo_score,
    'sft_delta': sft_score - base_score,
    'grpo_delta_over_sft': grpo_score - sft_score,
    'total_delta': grpo_score - base_score,
    'sft_best_checkpoint': sft_config.get('best_checkpoint'),
    'sft_best_eval_loss': sft_config.get('best_metric'),
    'grpo_zero_gradient_steps': zero_gradient_steps,
    'grpo_mean_clipped_ratio': mean_clipped_ratio,
    'scope_note': (
        'Custom disjoint 22-task test split; not an official HumanEval '
        'leaderboard score.'
    ),
}
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
(BUNDLE_DIR / 'comparison.json').write_text(json.dumps(comparison, indent=2) + '\n')
report = (
    '# Base vs SFT vs SFT+GRPO\n\n'
    f'- Model: `{MODEL}`\n'
    f'- Git commit: `{commit}`\n'
    f'- Training seed: {SEED}\n'
    f'- Held-out tasks: {base["tasks"]}\n'
    f'- Base pass@1: {base_score:.4f}\n'
    f'- SFT pass@1: {sft_score:.4f} ({sft_score - base_score:+.4f})\n'
    f'- SFT+GRPO pass@1: {grpo_score:.4f} ({grpo_score - sft_score:+.4f} over SFT)\n'
    f'- GRPO zero-gradient steps: {zero_gradient_steps}/{len(grpo_logs)}\n'
    f'- GRPO mean clipped ratio: {mean_clipped_ratio:.4f}\n\n'
    'The test split was not used by SFT or GRPO. Results are custom experiment '
    'metrics, not an official leaderboard score.\n'
)
(BUNDLE_DIR / 'REPORT.md').write_text(report)
shutil.copytree(BASELINE_DIR, BUNDLE_DIR / 'base-results', dirs_exist_ok=True)
shutil.copytree(SFT_RESULTS, BUNDLE_DIR / 'sft-results', dirs_exist_ok=True)
shutil.copytree(GRPO_RESULTS, BUNDLE_DIR / 'sft-grpo-results', dirs_exist_ok=True)
copy_final_adapter(SFT_DIR, BUNDLE_DIR / 'sft-adapter')
copy_final_adapter(GRPO_DIR, BUNDLE_DIR / 'sft-grpo-adapter')

print('STEP 10/10 - Packaging and downloading the experiment', flush=True)
archive = shutil.make_archive(
    f'/content/code-sft-grpo-seed-{SEED}-results', 'zip', BUNDLE_DIR
)
print('\nEXPERIMENT SUCCESSFUL')
print(f'Base pass@1:      {base_score:.4f}')
print(f'SFT pass@1:       {sft_score:.4f}')
print(f'SFT+GRPO pass@1:  {grpo_score:.4f}')
print(f'Total difference: {grpo_score - base_score:+.4f}')
print(f'Downloadable archive: {archive}')
colab_files.download(archive)
